# Prediksi Tinggi Muka Air Sungai

Pipeline end-to-end untuk audit data, feature engineering lingkungan dan HydroRIVERS, rolling-origin validation, ensemble CatBoost, dan pembuatan submission.

**Guard training:** notebook memakai `MODE = "audit"` secara default. Menjalankan seluruh cell tidak akan melakukan fitting sampai mode diubah secara eksplisit. Desain dan hasil audit lengkap ada di `architecture.md` dan `logs.md`.


## 1. Import, path, dan konfigurasi model


In [ ]:
"""Training and submission pipeline for river water-level (TMA) forecasting.

The pipeline is deliberately non-recursive: every time-varying future feature is
available in ``data_lingkungan.csv`` or is static/calendar information.  The only
target-derived feature is a leakage-safe seasonal baseline fitted on past labels.
This avoids compounding autoregressive error over the 242-day test
horizon.

Training is never started implicitly. The execution cell defaults to audit mode.
"""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import logging
import math
import platform
import re
import sys
import traceback
import zipfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error


LOGGER = logging.getLogger("tma_forecasting")
KEYS = ["datetime", "nama_pos"]
NOTEBOOK_SOURCE_SHA256 = "b5866cbafb6461fd23af693cec95ecd305300ba1d7f08a55dad77a1b5df7acfb"
ID_PATTERN = re.compile(
    r"^(?P<datetime>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (?P<nama_pos>.+)$"
)


@dataclass(frozen=True)
class Paths:
    root: Path = Path("datasets")
    artifacts: Path = Path("artifacts")

    @property
    def train(self) -> Path:
        return self.root / "train.csv"

    @property
    def test(self) -> Path:
        return self.root / "test.csv"

    @property
    def sample_submission(self) -> Path:
        return self.root / "sample_submission.csv"

    @property
    def coordinates(self) -> Path:
        return self.root / "data_pendukung" / "koordinat_pos.csv"

    @property
    def environment(self) -> Path:
        return self.root / "data_pendukung" / "data_lingkungan.csv"

    @property
    def rivers(self) -> Path:
        return (
            self.root
            / "data_pendukung"
            / "HydroRIVERS_v10_au_shp"
            / "HydroRIVERS_v10_au.shp"
        )


@dataclass(frozen=True)
class ModelConfig:
    task_type: str = "CPU"
    devices: str | None = None
    gpu_ram_part: float = 0.85
    expected_gpu_count: int = 2
    boosting_type: str = "Plain"
    border_count: int = 254
    metric_period: int = 5
    seeds: tuple[int, ...] = (17, 41, 83)
    default_iterations: int = 1800
    learning_rate: float = 0.035
    depth: int = 8
    l2_leaf_reg: float = 10.0
    huber_delta: float = 1.5
    early_stopping_rounds: int = 200
    verbose_every: int = 100
    upstream_decay_km: float = 75.0
    snapshot_interval_seconds: int = 300
    seasonal_bandwidth_days: float = 21.0
    seasonal_recency_half_life_years: float = 1.5
    seasonal_crossfit_folds: int = 5
    quality_neighbor_gap_days: int = 45
    blend_grid_size: int = 11


FOLDS = (
    ("sep_2023", "2023-09-18 18:00:00", "2023-09-19 06:00:00", "2024-05-18 18:00:00"),
    ("may_2024", "2024-05-18 18:00:00", "2024-05-19 06:00:00", "2025-01-18 18:00:00"),
    ("sep_2024", "2024-09-18 18:00:00", "2024-09-19 06:00:00", "2025-05-18 18:00:00"),
    ("jan_2025", "2025-01-18 18:00:00", "2025-01-19 06:00:00", "2025-09-18 18:00:00"),
)


RIVER_NUMERIC = [
    "LENGTH_KM",
    "DIST_DN_KM",
    "DIST_UP_KM",
    "CATCH_SKM",
    "UPLAND_SKM",
    "DIS_AV_CMS",
    "ORD_STRA",
    "ORD_CLAS",
    "ORD_FLOW",
]

RAW_STATE_COLUMNS = [
    "soil_moisture_0_7cm",
    "soil_moisture_7_28cm",
    "soil_moisture_28_100cm",
    "soil_moisture_100_255cm",
    "surface_pressure_hpa",
    "pressure_msl_hpa",
    "rmm1",
    "rmm2",
    "mjo_phase",
    "mjo_amplitude",
    "mjo_active",
    "nino_34",
]

CATEGORICAL_COLUMNS = [
    "nama_pos",
    "landcover_name",
    "hour_cat",
    "month_cat",
    "mjo_phase_cat",
    "main_riv_cat",
    "hybas_l12_cat",
    "landcover_class_cat",
    "endorheic_cat",
]

NON_FEATURE_COLUMNS = {
    "datetime",
    "id",
    "tma_mdpl",
    "_row_order",
    "mjo_phase",
    "landcover_class",
    "HYRIV_ID",
    "NEXT_DOWN",
    "MAIN_RIV",
    "HYBAS_L12",
    "ENDORHEIC",
    "wind_direction_deg",
}


## 2. Parsing ID, pemuatan data, dan audit schema

Missing baris target tidak dibuat menjadi pseudo-label. Semua key, station set, jam, dan urutan sample submission diperiksa sebelum feature engineering.


In [ ]:
def parse_submission_ids(frame: pd.DataFrame) -> pd.DataFrame:
    """Parse IDs without splitting station names that themselves contain ' - '."""
    parsed = frame["id"].str.extract(ID_PATTERN)
    if parsed.isna().any(axis=None):
        bad = frame.loc[parsed.isna().any(axis=1), "id"].head(3).tolist()
        raise ValueError(f"Invalid submission IDs; examples: {bad}")
    result = frame.copy()
    result["datetime"] = pd.to_datetime(parsed["datetime"], errors="raise")
    result["nama_pos"] = parsed["nama_pos"]
    result["_row_order"] = np.arange(len(result), dtype=np.int64)
    return result


def load_inputs(paths: Paths) -> tuple[pd.DataFrame, ...]:
    train = pd.read_csv(paths.train, parse_dates=["datetime"])
    test_raw = pd.read_csv(paths.test)
    sample = pd.read_csv(paths.sample_submission)
    coordinates = pd.read_csv(paths.coordinates)
    environment = pd.read_csv(paths.environment, parse_dates=["datetime"])
    test = parse_submission_ids(test_raw)
    return train, test, sample, coordinates, environment


def audit_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
) -> dict:
    """Fail early on silent schema, key, time, or station-set changes."""
    expected_train = {"datetime", "nama_pos", "tma_mdpl"}
    if set(train.columns) != expected_train:
        raise ValueError(f"Unexpected train schema: {train.columns.tolist()}")
    if train.duplicated(KEYS).any() or environment.duplicated(KEYS).any():
        raise ValueError("Duplicate (datetime, nama_pos) keys detected")
    if train["tma_mdpl"].isna().any():
        raise ValueError("Target contains explicit NaN values")
    if test["id"].duplicated().any():
        raise ValueError("Duplicate test IDs detected")
    if not test["id"].reset_index(drop=True).equals(sample["id"].reset_index(drop=True)):
        raise ValueError("test.csv and sample_submission.csv IDs/order differ")

    train_stations = set(train["nama_pos"])
    station_sets = {
        "test": set(test["nama_pos"]),
        "coordinates": set(coordinates["nama_pos"]),
        "environment": set(environment["nama_pos"]),
    }
    for label, stations in station_sets.items():
        if stations != train_stations:
            raise ValueError(
                f"Station mismatch for {label}: missing={sorted(train_stations-stations)}, "
                f"extra={sorted(stations-train_stations)}"
            )

    expected_hours = {6, 12, 18}
    if set(train["datetime"].dt.hour) - expected_hours:
        raise ValueError("Train contains target hours outside 06, 12, 18")
    if set(test["datetime"].dt.hour) != expected_hours:
        raise ValueError("Test hours are not exactly 06, 12, 18")

    env_counts = environment.groupby("nama_pos", observed=True).size()
    expected_env_rows = len(
        pd.date_range(environment["datetime"].min(), environment["datetime"].max(), freq="h")
    )
    if not (env_counts == expected_env_rows).all():
        raise ValueError("Environment is not a complete hourly grid for every station")

    return {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "environment_rows": int(len(environment)),
        "stations": int(len(train_stations)),
        "train_start": str(train["datetime"].min()),
        "train_end": str(train["datetime"].max()),
        "test_start": str(test["datetime"].min()),
        "test_end": str(test["datetime"].max()),
        "environment_missing_cells": int(environment.isna().sum().sum()),
        "solar_minus_999_cells": int((environment["solar_radiation_mj_m2"] <= -900).sum()),
    }


## 3. HydroRIVERS dan graf upstream

Koordinat pos dicocokkan dalam UTM 49S. Traversal `NEXT_DOWN` membentuk bobot lingkungan upstream yang menurun terhadap jarak hidraulik.


In [ ]:
def _read_local_rivers(path: Path, coordinates: pd.DataFrame) -> gpd.GeoDataFrame:
    pad = 0.75
    bbox = (
        coordinates["longitude"].min() - pad,
        coordinates["latitude"].min() - pad,
        coordinates["longitude"].max() + pad,
        coordinates["latitude"].max() + pad,
    )
    rivers = gpd.read_file(path, bbox=bbox)
    required = {
        "HYRIV_ID",
        "NEXT_DOWN",
        "MAIN_RIV",
        "HYBAS_L12",
        "ENDORHEIC",
        *RIVER_NUMERIC,
        "geometry",
    }
    missing = required - set(rivers.columns)
    if missing:
        raise ValueError(f"HydroRIVERS is missing columns: {sorted(missing)}")
    if rivers.crs is None:
        raise ValueError("HydroRIVERS has no CRS")
    return rivers


def match_stations_to_rivers(
    coordinates: pd.DataFrame, rivers: gpd.GeoDataFrame
) -> pd.DataFrame:
    points = gpd.GeoDataFrame(
        coordinates.copy(),
        geometry=gpd.points_from_xy(coordinates["longitude"], coordinates["latitude"]),
        crs="EPSG:4326",
    )
    # All supplied gauges are inside UTM zone 49S (108E-114E).
    points_utm = points.to_crs("EPSG:32749")
    rivers_utm = rivers.to_crs("EPSG:32749")
    nearest = gpd.sjoin_nearest(
        points_utm,
        rivers_utm,
        how="left",
        distance_col="river_distance_m",
    )
    nearest = (
        nearest.sort_values("river_distance_m")
        .loc[lambda x: ~x.index.duplicated(keep="first")]
        .sort_index()
    )
    if nearest["HYRIV_ID"].isna().any():
        raise ValueError("At least one station could not be matched to HydroRIVERS")

    keep = [
        "nama_pos",
        "latitude",
        "longitude",
        "river_distance_m",
        "HYRIV_ID",
        "NEXT_DOWN",
        "MAIN_RIV",
        "HYBAS_L12",
        "ENDORHEIC",
        *RIVER_NUMERIC,
    ]
    static = pd.DataFrame(nearest[keep]).reset_index(drop=True)
    static["river_match_confidence"] = np.exp(-static["river_distance_m"] / 1000.0)
    for col in ["CATCH_SKM", "UPLAND_SKM", "DIS_AV_CMS", "LENGTH_KM"]:
        static[f"log1p_{col.lower()}"] = np.log1p(static[col].clip(lower=0))
    static["main_riv_cat"] = static["MAIN_RIV"].astype("Int64").astype(str)
    static["hybas_l12_cat"] = static["HYBAS_L12"].astype("Int64").astype(str)
    static["endorheic_cat"] = static["ENDORHEIC"].astype("Int64").astype(str)
    return static


def build_upstream_weights(
    static: pd.DataFrame,
    rivers: gpd.GeoDataFrame,
    decay_km: float,
) -> pd.DataFrame:
    """Create target-by-source weights from exact NEXT_DOWN reach traversal."""
    stations = static["nama_pos"].tolist()
    reach_by_station = dict(zip(static["nama_pos"], static["HYRIV_ID"].astype(int)))
    dist_by_station = dict(zip(static["nama_pos"], static["DIST_DN_KM"].astype(float)))
    station_by_reach = {reach: station for station, reach in reach_by_station.items()}
    next_down = dict(zip(rivers["HYRIV_ID"].astype(int), rivers["NEXT_DOWN"].astype(int)))

    weights = pd.DataFrame(0.0, index=stations, columns=stations)
    for source in stations:
        weights.loc[source, source] = 1.0
        current = reach_by_station[source]
        seen = {current}
        for _ in range(len(next_down) + 1):
            current = next_down.get(current, 0)
            if current == 0 or current in seen:
                break
            seen.add(current)
            target = station_by_reach.get(current)
            if target is not None:
                distance = max(0.0, dist_by_station[source] - dist_by_station[target])
                weights.loc[target, source] = max(
                    weights.loc[target, source], math.exp(-distance / decay_km)
                )
    return weights.div(weights.sum(axis=1), axis=0)


def _add_upstream_environment(
    environment: pd.DataFrame,
    weights: pd.DataFrame,
    source_columns: Iterable[str],
) -> pd.DataFrame:
    result = environment
    station_order = weights.columns.tolist()
    for column in source_columns:
        pivot = result.pivot(index="datetime", columns="nama_pos", values=column).reindex(
            columns=station_order
        )
        upstream = pd.DataFrame(
            pivot.to_numpy(dtype=float) @ weights.to_numpy(dtype=float).T,
            index=pivot.index,
            columns=weights.index,
        )
        long = (
            upstream.rename_axis(columns="nama_pos")
            .stack(future_stack=True)
            .rename(f"upstream_{column}")
            .reset_index()
        )
        result = result.merge(long, on=KEYS, how="left", validate="one_to_one")
    return result


## 4. Sanitasi lingkungan dan fitur antecedent

Menangani sentinel radiasi `-999`, missing tail, arah angin siklik, hujan/soil upstream, rolling window, dan antecedent precipitation index secara kausal.


In [ ]:
def prepare_environment(
    environment: pd.DataFrame, upstream_weights: pd.DataFrame
) -> pd.DataFrame:
    env = environment.sort_values(KEYS).copy()

    # These columns are byte-identical in the supplied data; retain one copy only.
    if not env["rainfall_mm"].equals(env["rainfall_openmeteo_mm"]):
        raise ValueError("Rainfall duplicate columns have unexpectedly diverged")
    env = env.drop(columns="rainfall_openmeteo_mm")

    # -999 is an undocumented sentinel, not physically possible solar radiation.
    env["solar_radiation_missing"] = (
        env["solar_radiation_mj_m2"].isna()
        | (env["solar_radiation_mj_m2"] <= -900)
    ).astype("int8")
    env.loc[env["solar_radiation_mj_m2"] <= -900, "solar_radiation_mj_m2"] = np.nan

    # Missing indicators are retained even after causal forward filling.
    for col in RAW_STATE_COLUMNS:
        env[f"{col}_missing"] = env[col].isna().astype("int8")

    by_station = env.groupby("nama_pos", sort=False, observed=True)
    short_state = [c for c in RAW_STATE_COLUMNS if c != "nino_34"]
    env[short_state] = by_station[short_state].ffill(limit=24)
    # Nino 3.4 is a slowly varying monthly index; a 31-day causal hold is appropriate.
    env[["nino_34"]] = by_station[["nino_34"]].ffill(limit=31 * 24)

    radians = np.deg2rad(env["wind_direction_deg"])
    env["wind_direction_sin"] = np.sin(radians)
    env["wind_direction_cos"] = np.cos(radians)
    env["dewpoint_depression_c"] = env["temperature_c"] - env["dew_point_c"]

    env = _add_upstream_environment(
        env,
        upstream_weights,
        ["rainfall_mm", "soil_moisture_0_7cm"],
    )
    env = env.sort_values(KEYS).reset_index(drop=True)
    grouped = env.groupby("nama_pos", sort=False, observed=True)

    rolling_sum_specs = {
        "rainfall_mm": (6, 24, 72, 168, 336),
        "upstream_rainfall_mm": (6, 24, 72, 168, 336),
    }
    for col, windows in rolling_sum_specs.items():
        for hours in windows:
            env[f"{col}_sum_{hours}h"] = grouped[col].transform(
                lambda s, w=hours: s.rolling(w, min_periods=max(1, w // 4)).sum()
            )
        env[f"{col}_max_24h"] = grouped[col].transform(
            lambda s: s.rolling(24, min_periods=6).max()
        )
        for half_life in (12, 48, 168):
            env[f"{col}_api_halflife_{half_life}h"] = grouped[col].transform(
                lambda s, h=half_life: s.ewm(
                    halflife=h, adjust=False, min_periods=1
                ).mean()
            )

    rolling_mean_specs = {
        "soil_moisture_0_7cm": (24, 168),
        "soil_moisture_28_100cm": (24, 168),
        "upstream_soil_moisture_0_7cm": (24, 168),
        "temperature_c": (24, 72),
        "humidity_pct": (24, 72),
        "surface_pressure_hpa": (24,),
    }
    for col, windows in rolling_mean_specs.items():
        for hours in windows:
            env[f"{col}_mean_{hours}h"] = grouped[col].transform(
                lambda s, w=hours: s.rolling(
                    w, min_periods=max(1, w // 4)
                ).mean()
            )

    # Physical sanitation. Values are not fitted against the future distribution.
    env["rainfall_mm"] = env["rainfall_mm"].clip(lower=0)
    env["rainfall_max_24h_mm"] = env["rainfall_max_24h_mm"].clip(lower=0)
    env["solar_radiation_mj_m2"] = env["solar_radiation_mj_m2"].clip(lower=0)
    for col in [c for c in env.columns if "soil_moisture" in c]:
        env[col] = env[col].clip(0, 1)
    return env


## 5. Kalender, kategori, dan matriks fitur


In [ ]:
def add_calendar_and_categories(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    dt = out["datetime"]
    doy = dt.dt.dayofyear.astype(float)
    hour = dt.dt.hour.astype(float)
    out["dayofyear_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["dayofyear_cos"] = np.cos(2 * np.pi * doy / 365.25)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    out["is_wet_season"] = dt.dt.month.isin([11, 12, 1, 2, 3, 4]).astype("int8")
    out["hour_cat"] = dt.dt.hour.astype(str)
    out["month_cat"] = dt.dt.month.astype(str)
    out["mjo_phase_cat"] = out["mjo_phase"].round().astype("Int64").astype(str)
    out["landcover_class_cat"] = out["landcover_class"].astype("Int64").astype(str)
    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].fillna("__MISSING__").astype(str)
    return out


def build_feature_tables(
    train: pd.DataFrame,
    test: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
    paths: Paths,
    config: ModelConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rivers = _read_local_rivers(paths.rivers, coordinates)
    static = match_stations_to_rivers(coordinates, rivers)
    weights = build_upstream_weights(static, rivers, config.upstream_decay_km)
    env_features = prepare_environment(environment, weights)

    train_features = train.merge(
        env_features, on=KEYS, how="left", validate="many_to_one"
    )
    test_features = test.merge(
        env_features, on=KEYS, how="left", validate="many_to_one"
    )
    train_features = train_features.merge(
        static, on="nama_pos", how="left", validate="many_to_one"
    )
    test_features = test_features.merge(
        static, on="nama_pos", how="left", validate="many_to_one"
    )
    if (
        train_features["rainfall_mm"].isna().any()
        or test_features["rainfall_mm"].isna().any()
    ):
        raise ValueError("Some target timestamps have no matching environmental row")

    train_features = add_calendar_and_categories(train_features)
    test_features = add_calendar_and_categories(test_features)
    return train_features, test_features, weights


def feature_columns(frame: pd.DataFrame) -> list[str]:
    cols = [c for c in frame.columns if c not in NON_FEATURE_COLUMNS]
    # Target-derived features are intentionally forbidden.
    forbidden = [c for c in cols if c.startswith("tma_") or "target" in c.lower()]
    if forbidden:
        raise ValueError(f"Target leakage features detected: {forbidden}")
    return cols


## 6. Target robust, spike weighting, metrik, dan model

Target dinormalisasi robust per pos. Spike satu-titik hanya diturunkan bobotnya bila kedua tetangganya konsisten; kejadian tinggi berkelanjutan tetap berbobot penuh.


In [ ]:
def fit_target_stats(train: pd.DataFrame) -> pd.DataFrame:
    grouped = train.groupby("nama_pos", observed=True)["tma_mdpl"]
    stats = grouped.agg(
        center="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
    )
    stats["scale"] = ((stats["q75"] - stats["q25"]) / 1.349).clip(lower=0.05)
    return stats[["center", "scale"]]


def normalize_target(frame: pd.DataFrame, stats: pd.DataFrame) -> np.ndarray:
    center = frame["nama_pos"].map(stats["center"])
    scale = frame["nama_pos"].map(stats["scale"])
    if center.isna().any() or scale.isna().any():
        raise ValueError("Target normalization encountered an unseen station")
    return ((frame["tma_mdpl"] - center) / scale).to_numpy(dtype=float)


def denormalize_prediction(
    prediction: np.ndarray, stations: pd.Series, stats: pd.DataFrame
) -> np.ndarray:
    center = stations.map(stats["center"]).to_numpy(dtype=float)
    scale = stations.map(stats["scale"]).to_numpy(dtype=float)
    return center + scale * np.asarray(prediction, dtype=float)


def isolated_spike_weights(train: pd.DataFrame) -> tuple[np.ndarray, int]:
    """Downweight only one-step spikes whose two neighbors agree with each other."""
    ordered = train.sort_values(KEYS).copy()
    group = ordered.groupby("nama_pos", sort=False, observed=True)
    ordered["prev_y"] = group["tma_mdpl"].shift(1)
    ordered["next_y"] = group["tma_mdpl"].shift(-1)
    ordered["prev_t"] = group["datetime"].shift(1)
    ordered["next_t"] = group["datetime"].shift(-1)

    diff_scale = group["tma_mdpl"].transform(
        lambda s: (s.diff().abs().median() * 1.4826)
    ).clip(lower=0.05)
    station_stats = fit_target_stats(ordered)
    robust_scale = ordered["nama_pos"].map(station_stats["scale"])
    midpoint = (ordered["prev_y"] + ordered["next_y"]) / 2.0
    threshold = np.maximum(12.0 * diff_scale, 8.0 * robust_scale)
    neighbors_agree = (ordered["prev_y"] - ordered["next_y"]).abs() <= np.maximum(
        4.0 * diff_scale, 2.0 * robust_scale
    )
    adjacent = (
        ((ordered["datetime"] - ordered["prev_t"]) <= pd.Timedelta(hours=12))
        & ((ordered["next_t"] - ordered["datetime"]) <= pd.Timedelta(hours=12))
    )
    spike = adjacent & neighbors_agree & ((ordered["tma_mdpl"] - midpoint).abs() > threshold)
    ordered["weight"] = np.where(spike, 0.05, 1.0)
    weights = ordered["weight"].reindex(train.index).to_numpy(dtype=float)
    return weights, int(spike.sum())


def metrics_frame(y_true: np.ndarray, y_pred: np.ndarray, stations: pd.Series) -> dict:
    errors = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    station_mae = pd.DataFrame({"station": stations.to_numpy(), "ae": errors}).groupby(
        "station", observed=True
    )["ae"].mean()
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "macro_station_mae": float(station_mae.mean()),
        "worst_station_mae": float(station_mae.max()),
        "p95_absolute_error": float(np.quantile(errors, 0.95)),
    }


def seasonal_baseline(train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    reference = train.assign(
        month=train["datetime"].dt.month,
        hour=train["datetime"].dt.hour,
    )
    lookup = reference.groupby(["nama_pos", "month", "hour"], observed=True)[
        "tma_mdpl"
    ].median()
    fallback = reference.groupby("nama_pos", observed=True)["tma_mdpl"].median()
    key = pd.MultiIndex.from_arrays(
        [valid["nama_pos"], valid["datetime"].dt.month, valid["datetime"].dt.hour]
    )
    pred = lookup.reindex(key).to_numpy(dtype=float)
    missing = np.isnan(pred)
    pred[missing] = valid.loc[missing, "nama_pos"].map(fallback).to_numpy(dtype=float)
    return pred


def catboost_accelerator_params(config: ModelConfig) -> dict:
    """Return explicit CPU/GPU settings shared by every model variant."""
    task_type = config.task_type.upper()
    if task_type not in {"CPU", "GPU"}:
        raise ValueError("task_type must be either CPU or GPU")
    boosting_type = config.boosting_type.title()
    if boosting_type not in {"Plain", "Ordered"}:
        raise ValueError("boosting_type must be Plain or Ordered")
    if not 1 <= config.border_count <= 65535:
        raise ValueError("border_count must be between 1 and 65535")
    if config.metric_period < 1:
        raise ValueError("metric_period must be at least 1")
    if config.verbose_every % config.metric_period != 0:
        raise ValueError("verbose_every must be a multiple of metric_period")
    params = {
        "task_type": task_type,
        "boosting_type": boosting_type,
        "border_count": config.border_count,
        "metric_period": config.metric_period,
    }
    if task_type == "GPU":
        if not config.devices:
            raise ValueError("devices must be set when task_type='GPU'")
        params.update(
            devices=config.devices,
            gpu_ram_part=config.gpu_ram_part,
            data_partition="DocParallel",
        )
    return params


def validate_accelerator(config: ModelConfig) -> dict:
    """Fail early when a requested Kaggle multi-GPU runtime is unavailable."""
    backend = {
        "task_type": config.task_type.upper(),
        "devices": config.devices,
        "boosting_type": config.boosting_type.title(),
        "border_count": config.border_count,
        "metric_period": config.metric_period,
    }
    if config.task_type.upper() != "GPU":
        return {**backend, "devices": None, "gpu_count": 0}
    from catboost.utils import get_gpu_device_count

    gpu_count = int(get_gpu_device_count())
    if gpu_count < config.expected_gpu_count:
        raise RuntimeError(
            f"This run requires {config.expected_gpu_count} GPUs, but CatBoost detected "
            f"{gpu_count}. In Kaggle Settings, select GPU T4 x2."
        )
    return {**backend, "gpu_count": gpu_count}


def make_model(
    config: ModelConfig,
    seed: int,
    iterations: int,
    train_dir: Path | None = None,
):
    try:
        from catboost import CatBoostRegressor
    except ImportError as exc:
        raise RuntimeError(
            "CatBoost is required for training. Install requirements.txt first."
        ) from exc
    params = dict(
        iterations=iterations,
        learning_rate=config.learning_rate,
        depth=config.depth,
        loss_function=f"Huber:delta={config.huber_delta}",
        eval_metric="MAE",
        l2_leaf_reg=config.l2_leaf_reg,
        random_seed=seed,
        random_strength=0.8,
        bootstrap_type="Bayesian",
        bagging_temperature=0.7,
        allow_writing_files=train_dir is not None,
        thread_count=-1,
        verbose=config.verbose_every,
    )
    params.update(catboost_accelerator_params(config))
    if train_dir is not None:
        train_dir.mkdir(parents=True, exist_ok=True)
        params.update(
            train_dir=str(train_dir),
            save_snapshot=True,
            snapshot_file="training.snapshot",
            snapshot_interval=config.snapshot_interval_seconds,
        )
    return CatBoostRegressor(**params)


def _xy(
    frame: pd.DataFrame, columns: Sequence[str]
) -> pd.DataFrame:
    result = frame.loc[:, columns].copy()
    for col in set(columns) & set(CATEGORICAL_COLUMNS):
        result[col] = result[col].fillna("__MISSING__").astype(str)
    return result


## 7. Run tracking dan handoff untuk penyempurnaan model

Setiap training membuat direktori dan ZIP bertimestamp UTC. Arsip `in_progress` diperbarui selama training; arsip `complete` atau `failed` menjadi handoff akhir untuk analisis Codex.


In [ ]:
HANDOFF_SCHEMA_VERSION = "1.1"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def notebook_provenance(notebook_path: Path | None) -> dict:
    candidates = []
    if notebook_path is not None:
        candidates.append(Path(notebook_path))
    candidates.extend([Path.cwd() / "notebook-150726.ipynb", Path("notebook-150726.ipynb")])
    checked = []
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if str(resolved) in checked:
            continue
        checked.append(str(resolved))
        if resolved.is_file():
            notebook_payload = json.loads(resolved.read_text(encoding="utf-8"))
            source_payload = [
                {"cell_type": cell.get("cell_type"), "id": cell.get("id"),
                 "source": [line for line in cell.get("source", [])
                            if not line.strip().startswith("NOTEBOOK_SOURCE_SHA256 =")]}
                for cell in notebook_payload.get("cells", [])
            ]
            source_bytes = json.dumps(
                source_payload, sort_keys=True, separators=(",", ":"),
                ensure_ascii=False,
            ).encode("utf-8")
            source_sha256 = hashlib.sha256(source_bytes).hexdigest()
            if source_sha256 != NOTEBOOK_SOURCE_SHA256:
                raise RuntimeError("Notebook source hash does not match declared provenance")
            return {
                "status": "hashed", "path": str(resolved),
                "sha256": sha256_file(resolved),
                "source_sha256": source_sha256,
                "size_bytes": int(resolved.stat().st_size),
            }
    return {"status": "declared_source_hash_only", "path": None,
            "sha256": None, "source_sha256": NOTEBOOK_SOURCE_SHA256,
            "size_bytes": None, "checked_paths": checked}


def effective_v2_model_manifest(
    config: ModelConfig, tuning: dict, tuning_source: str,
) -> dict:
    alpha = float(tuning["blend_rmse_fraction"])
    beta = float(tuning["blend_residual_fraction"])
    return {
        "architecture": "direct Huber anchor + seasonal residual Huber/RMSE blend",
        "tuning_source": tuning_source,
        "blend": {
            "rmse_fraction_within_residual": alpha,
            "residual_fraction_in_final": beta,
            "effective_component_weights": {
                "anchor": 1.0 - beta,
                "residual_huber": beta * (1.0 - alpha),
                "residual_rmse": beta * alpha,
            },
        },
        "iterations": {
            "anchor": int(tuning["anchor_iterations"]),
            "residual_huber": int(tuning["huber_iterations"]),
            "residual_rmse": int(tuning["rmse_iterations"]),
        },
        "seeds": [int(seed) for seed in config.seeds],
        "selection_rule": tuning.get("selection_rule"),
    }


def core_artifact_hashes(run_dir: Path) -> dict:
    relative_paths = ["config.json", "feature_columns.json", "v2_tuning.json",
                      "v2_model_metadata.json", "submission.csv"]
    return {relative: sha256_file(run_dir / relative)
            for relative in relative_paths if (run_dir / relative).is_file()}


def create_run_directory(output_root: Path, mode: str) -> Path:
    output_root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = output_root / f"handoff_{stamp}_{mode}"
    suffix = 1
    while run_dir.exists():
        run_dir = output_root / f"handoff_{stamp}_{mode}_{suffix:02d}"
        suffix += 1
    for child in ("data", "metrics", "models", "training_progress"):
        (run_dir / child).mkdir(parents=True, exist_ok=True)
    return run_dir


def update_manifest(run_dir: Path, **changes) -> dict:
    path = run_dir / "manifest.json"
    manifest = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    manifest.update(changes)
    manifest["updated_at_utc"] = utc_now_iso()
    write_json(path, manifest)
    return manifest


def attach_run_logger(run_dir: Path) -> None:
    for handler in list(LOGGER.handlers):
        if getattr(handler, "_tma_handoff_handler", False):
            LOGGER.removeHandler(handler)
            handler.close()
    handler = logging.FileHandler(run_dir / "run.log", encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    handler._tma_handoff_handler = True
    LOGGER.addHandler(handler)
    LOGGER.setLevel(logging.INFO)


def runtime_versions() -> dict:
    packages = ["catboost", "geopandas", "numpy", "pandas", "pyarrow", "scikit-learn", "shapely"]
    versions = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
    }


def input_inventory(paths: Paths) -> list[dict]:
    river_sidecars = sorted(paths.rivers.parent.glob(f"{paths.rivers.stem}.*"))
    files = [paths.train, paths.test, paths.sample_submission, paths.coordinates, paths.environment, *river_sidecars]
    return [
        {"path": str(path), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else None}
        for path in files
    ]


def save_frame(frame: pd.DataFrame, path_without_suffix: Path) -> Path:
    """Prefer compressed parquet; fall back to gzip CSV if pyarrow is absent."""
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = path_without_suffix.with_suffix(".parquet")
    try:
        frame.to_parquet(parquet_path, index=False, compression="zstd")
        return parquet_path
    except (ImportError, ModuleNotFoundError):
        csv_path = path_without_suffix.with_suffix(".csv.gz")
        frame.to_csv(csv_path, index=False, compression="gzip")
        return csv_path


def environment_drift_report(
    train_features: pd.DataFrame, test_features: pd.DataFrame, columns: list[str]
) -> pd.DataFrame:
    numeric = [c for c in columns if pd.api.types.is_numeric_dtype(train_features[c])]
    rows = []
    for col in numeric:
        train_values = train_features[col].replace([np.inf, -np.inf], np.nan)
        test_values = test_features[col].replace([np.inf, -np.inf], np.nan)
        train_std = float(train_values.std())
        rows.append({
            "feature": col,
            "train_missing_pct": float(train_values.isna().mean() * 100),
            "test_missing_pct": float(test_values.isna().mean() * 100),
            "train_mean": float(train_values.mean()),
            "test_mean": float(test_values.mean()),
            "train_std": train_std,
            "test_std": float(test_values.std()),
            "standardized_mean_shift": float((test_values.mean() - train_values.mean()) / train_std) if train_std > 0 else np.nan,
            "train_q01": float(train_values.quantile(0.01)),
            "train_q50": float(train_values.quantile(0.50)),
            "train_q99": float(train_values.quantile(0.99)),
            "test_q01": float(test_values.quantile(0.01)),
            "test_q50": float(test_values.quantile(0.50)),
            "test_q99": float(test_values.quantile(0.99)),
        })
    return pd.DataFrame(rows)


def write_handoff_readme(run_dir: Path) -> None:
    text = """# TMA model handoff\n\nThis archive is a reproducible diagnostic handoff for the next model iteration.\n\nCore files:\n- manifest.json: run identity, timestamps, status, and stage.\n- config.json / runtime_versions.json / input_inventory.json: reproducibility context.\n- data/train_features.* and test_features.*: exact model matrices when enabled.\n- data/train_sample_diagnostics.*: normalized target, sample weights, and spike flags.\n- metrics/oof_predictions.*: leakage-safe validation predictions and residuals.\n- metrics/validation_metrics.csv: global fold scores and baseline comparison.\n- metrics/fold_station_metrics.csv: failure analysis by station.\n- metrics/residual_slices.csv: residuals by station, month, and hour.\n- metrics/feature_importance.csv: importance by fold/model.\n- metrics/environment_drift.csv: train-to-test feature distribution shift.\n- training_progress/: live CatBoost logs and snapshots for every fold/seed.\n- models/: fold and final CatBoost models.\n- submission.csv: exact competition output when final fit ran.\n- run.log / error_traceback.txt: orchestration log and failure context.\n\nUse OOF predictions—not training error—as the primary basis for refinements.\n"""
    text += """
Manifest schema 1.1 records notebook SHA-256, effective runtime configuration, expanded component weights, per-component iteration counts, tuning source, and hashes of core output artifacts.
"""
    text += """
V2 additions:
- v2_tuning.json: OOF-selected component weights and iterations.
- data/v2_quality_diagnostics.csv: keys, targets, quality weights, and reasons.
- data/v2_seasonal_lookup.csv: final seasonal baseline lookup.
- metrics/v2_oof_predictions.csv: direct anchor, both residual models, and final blend.
- metrics/v2_validation_metrics.csv / v2_blend_search.csv: fold scores and joint grid.
- metrics/v2_feature_importance.csv: importance per component and fold.
- models/v2_*.cbm: validation and final component models.
"""
    (run_dir / "HANDOFF_README.md").write_text(text, encoding="utf-8")


def create_handoff_zip(run_dir: Path, status: str) -> Path:
    if status not in {"in_progress", "complete", "failed"}:
        raise ValueError("Invalid handoff ZIP status")
    destination = run_dir.parent / f"{run_dir.name}_{status}.zip"
    temporary = destination.with_suffix(".zip.tmp")
    if temporary.exists():
        temporary.unlink()
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(run_dir.rglob("*")):
            if path.is_file():
                already_compressed = path.suffix.lower() in {".parquet", ".gz", ".cbm", ".snapshot"}
                archive.write(
                    path, arcname=str(Path(run_dir.name) / path.relative_to(run_dir)),
                    compress_type=zipfile.ZIP_STORED if already_compressed else zipfile.ZIP_DEFLATED,
                )
    temporary.replace(destination)
    if status in {"complete", "failed"}:
        in_progress = run_dir.parent / f"{run_dir.name}_in_progress.zip"
        if in_progress.exists():
            in_progress.unlink()
    return destination


def save_pretraining_handoff(
    run_dir: Path,
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    upstream_weights: pd.DataFrame,
    save_feature_matrices: bool,
) -> None:
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    write_json(run_dir / "feature_columns.json", {"features": columns, "categorical": cat_columns})
    upstream_weights.to_csv(run_dir / "data" / "upstream_station_weights.csv")
    stats = fit_target_stats(train_features)
    stats.to_csv(run_dir / "data" / "target_normalization.csv")
    sample_weights, _ = isolated_spike_weights(train_features)
    diagnostics = train_features[KEYS + ["tma_mdpl"]].copy()
    diagnostics["target_normalized"] = normalize_target(train_features, stats)
    diagnostics["sample_weight"] = sample_weights
    diagnostics["isolated_spike_flag"] = (sample_weights < 1).astype("int8")
    save_frame(diagnostics, run_dir / "data" / "train_sample_diagnostics")
    drift = environment_drift_report(train_features, test_features, columns)
    drift.to_csv(run_dir / "metrics" / "environment_drift.csv", index=False)
    if save_feature_matrices:
        train_export_columns = list(dict.fromkeys(KEYS + ["tma_mdpl"] + columns))
        test_export_columns = list(dict.fromkeys(["id"] + KEYS + columns))
        train_export = train_features[train_export_columns].copy()
        test_export = test_features[test_export_columns].copy()
        save_frame(train_export, run_dir / "data" / "train_features")
        save_frame(test_export, run_dir / "data" / "test_features")
    update_manifest(run_dir, stage="features_ready", feature_count=len(columns), status="in_progress")
    create_handoff_zip(run_dir, "in_progress")


## 8. Rolling-origin validation

Empat fold memakai horizon panjang agar menyerupai deployment 242 hari. Cell ini hanya mendefinisikan fungsi; belum menjalankan training.


In [ ]:
def run_validation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
) -> tuple[pd.DataFrame, int]:
    records: list[dict] = []
    best_iterations: list[int] = []
    oof_parts: list[pd.DataFrame] = []
    station_records: list[dict] = []
    importance_records: list[dict] = []
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]

    for fold_name, train_end, valid_start, valid_end in FOLDS:
        fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
        valid = train_features[
            train_features["datetime"].between(
                pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
            )
        ].copy()
        eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        fit = fit[fit["nama_pos"].isin(eligible)]
        valid = valid[valid["nama_pos"].isin(eligible)]
        if fit.empty or valid.empty:
            LOGGER.warning("Skipping empty fold %s", fold_name)
            continue

        stats = fit_target_stats(fit)
        y_fit = normalize_target(fit, stats)
        y_valid_z = normalize_target(valid, stats)
        sample_weights, spike_count = isolated_spike_weights(fit)
        progress_dir = artifacts / "training_progress" / f"fold_{fold_name}"
        model = make_model(
            config, seed=config.seeds[0], iterations=config.default_iterations,
            train_dir=progress_dir,
        )
        model.fit(
            _xy(fit, columns),
            y_fit,
            cat_features=cat_columns,
            sample_weight=sample_weights,
            eval_set=(_xy(valid, columns), y_valid_z),
            early_stopping_rounds=config.early_stopping_rounds,
            use_best_model=True,
        )
        pred_z = model.predict(_xy(valid, columns))
        pred = denormalize_prediction(pred_z, valid["nama_pos"], stats)
        model_metrics = metrics_frame(valid["tma_mdpl"].to_numpy(), pred, valid["nama_pos"])
        baseline_pred = seasonal_baseline(fit, valid)
        baseline_metrics = metrics_frame(
            valid["tma_mdpl"].to_numpy(), baseline_pred, valid["nama_pos"]
        )
        best_iteration = int(model.get_best_iteration()) + 1
        best_iterations.append(best_iteration)
        fold_model_path = artifacts / "models" / f"validation_{fold_name}.cbm"
        model.save_model(str(fold_model_path))
        fold_stats = stats.reset_index()
        fold_stats.to_csv(artifacts / "data" / f"target_stats_{fold_name}.csv", index=False)

        oof = valid[KEYS + ["tma_mdpl"]].copy()
        oof["fold"] = fold_name
        oof["target_normalized"] = y_valid_z
        oof["prediction_normalized"] = pred_z
        oof["prediction"] = pred
        oof["baseline_prediction"] = baseline_pred
        oof["error"] = oof["prediction"] - oof["tma_mdpl"]
        oof["absolute_error"] = oof["error"].abs()
        oof_parts.append(oof)
        for station, station_frame in oof.groupby("nama_pos", observed=True):
            station_records.append({
                "fold": fold_name,
                "nama_pos": station,
                "rows": len(station_frame),
                "mae": float(station_frame["absolute_error"].mean()),
                "rmse": float(np.sqrt(np.mean(station_frame["error"] ** 2))),
                "bias": float(station_frame["error"].mean()),
            })
        for feature, importance in zip(columns, model.get_feature_importance()):
            importance_records.append({
                "fold": fold_name, "feature": feature,
                "importance": float(importance),
            })
        records.append(
            {
                "fold": fold_name,
                "train_rows": len(fit),
                "valid_rows": len(valid),
                "valid_stations": valid["nama_pos"].nunique(),
                "isolated_spikes_downweighted": spike_count,
                "best_iteration": best_iteration,
                **{f"model_{k}": v for k, v in model_metrics.items()},
                **{f"baseline_{k}": v for k, v in baseline_metrics.items()},
            }
        )
        LOGGER.info("Completed fold %s: %s", fold_name, model_metrics)
        cumulative_oof = pd.concat(oof_parts, ignore_index=True)
        save_frame(cumulative_oof, artifacts / "metrics" / "oof_predictions")
        pd.DataFrame(records).to_csv(artifacts / "metrics" / "validation_metrics.csv", index=False)
        pd.DataFrame(station_records).to_csv(artifacts / "metrics" / "fold_station_metrics.csv", index=False)
        pd.DataFrame(importance_records).to_csv(artifacts / "metrics" / "feature_importance.csv", index=False)
        update_manifest(artifacts, stage=f"validation_{fold_name}_complete", completed_folds=[r["fold"] for r in records])
        create_handoff_zip(artifacts, "in_progress")

    if not records:
        raise RuntimeError("No validation fold was runnable")
    report = pd.DataFrame(records)
    recommended = int(np.median(best_iterations))
    artifacts.mkdir(parents=True, exist_ok=True)
    report.to_csv(artifacts / "metrics" / "validation_metrics.csv", index=False)
    oof_all = pd.concat(oof_parts, ignore_index=True)
    residual_slices = (
        oof_all.assign(
            month=oof_all["datetime"].dt.month,
            hour=oof_all["datetime"].dt.hour,
        )
        .groupby(["nama_pos", "month", "hour"], observed=True)
        .agg(
            rows=("error", "size"),
            mae=("absolute_error", "mean"),
            bias=("error", "mean"),
            p95_absolute_error=("absolute_error", lambda s: s.quantile(0.95)),
        )
        .reset_index()
    )
    residual_slices.to_csv(artifacts / "metrics" / "residual_slices.csv", index=False)
    (artifacts / "recommended_iterations.txt").write_text(
        f"{recommended}\n", encoding="utf-8"
    )
    update_manifest(artifacts, stage="validation_complete", recommended_iterations=recommended)
    create_handoff_zip(artifacts, "in_progress")
    return report, recommended


## 9. Final ensemble V1 (legacy/reproducibility)

Tiga seed dirata-ratakan pada skala target ternormalisasi. Cell ini hanya mendefinisikan fungsi.


In [ ]:
def fit_final_and_submit(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    sample: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    iterations: int,
) -> Path:
    artifacts.mkdir(parents=True, exist_ok=True)
    stats = fit_target_stats(train_features)
    y = normalize_target(train_features, stats)
    weights, spike_count = isolated_spike_weights(train_features)
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    predictions = []
    final_importance_records = []

    for seed in config.seeds:
        progress_dir = artifacts / "training_progress" / f"final_seed_{seed}"
        model = make_model(
            config, seed=seed, iterations=iterations, train_dir=progress_dir
        )
        model.fit(
            _xy(train_features, columns),
            y,
            cat_features=cat_columns,
            sample_weight=weights,
        )
        model.save_model(str(artifacts / "models" / f"catboost_seed_{seed}.cbm"))
        predictions.append(model.predict(_xy(test_features, columns)))
        for feature, importance in zip(columns, model.get_feature_importance()):
            final_importance_records.append({
                "seed": seed, "feature": feature,
                "importance": float(importance),
            })
        pd.DataFrame(final_importance_records).to_csv(
            artifacts / "metrics" / "final_feature_importance.csv", index=False
        )
        update_manifest(
            artifacts, stage=f"final_seed_{seed}_complete",
            completed_final_seeds=[int(s) for s in config.seeds if (artifacts / "models" / f"catboost_seed_{s}.cbm").exists()],
        )
        create_handoff_zip(artifacts, "in_progress")

    mean_z = np.mean(predictions, axis=0)
    prediction = denormalize_prediction(mean_z, test_features["nama_pos"], stats)
    ordered = pd.DataFrame(
        {
            "_row_order": test_features["_row_order"],
            "id": test_features["id"],
            "tma_mdpl": prediction,
        }
    ).sort_values("_row_order")
    submission = sample[["id"]].merge(
        ordered[["id", "tma_mdpl"]], on="id", how="left", validate="one_to_one"
    )
    if submission["tma_mdpl"].isna().any() or not np.isfinite(submission["tma_mdpl"]).all():
        raise ValueError("Submission contains missing/non-finite predictions")
    output_path = artifacts / "submission.csv"
    submission.to_csv(output_path, index=False)

    stats.to_csv(artifacts / "data" / "target_normalization.csv")
    (artifacts / "feature_columns.json").write_text(
        json.dumps({"features": columns, "categorical": cat_columns}, indent=2),
        encoding="utf-8",
    )
    metadata = {
        "iterations": iterations,
        "seeds": list(config.seeds),
        "isolated_spikes_downweighted": spike_count,
        "train_rows": len(train_features),
        "test_rows": len(test_features),
        "model_config": asdict(config),
    }
    (artifacts / "run_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    update_manifest(artifacts, stage="final_fit_complete", final_iterations=iterations)
    create_handoff_zip(artifacts, "in_progress")
    return output_path


## 10. Model V2: direct anchor, baseline residual cross-fit, dan dual objective

V2 mempertahankan direct Huber sebagai anchor, lalu memodelkan residual terhadap baseline musiman per pos/jam dengan Huber dan RMSE. Baseline training dihitung cross-fit sehingga target baris tersebut tidak ikut membentuk baseline-nya. Bobot model RMSE disejajarkan ke squared error meter dan seluruh bobot blend dipilih hanya dari rolling OOF.


In [ ]:
def quality_diagnostics_v2(frame: pd.DataFrame, config: ModelConfig) -> pd.DataFrame:
    """Downweight likely sensor faults while retaining sustained hydrological events."""
    ordered = frame.sort_values(KEYS).copy()
    group = ordered.groupby("nama_pos", sort=False, observed=True)
    stats = fit_target_stats(ordered)
    scale = ordered["nama_pos"].map(stats["scale"]).astype(float)
    station_median = ordered["nama_pos"].map(stats["center"]).astype(float)
    diff_scale = group["tma_mdpl"].transform(
        lambda s: max(float(s.diff().abs().median() * 1.4826), 0.05)
    )
    offsets = {"prev1": 1, "prev2": 2, "next1": -1, "next2": -2}
    values = pd.DataFrame({name: group["tma_mdpl"].shift(offset) for name, offset in offsets.items()})
    times = pd.DataFrame({name: group["datetime"].shift(offset) for name, offset in offsets.items()})
    max_gap = pd.Timedelta(days=config.quality_neighbor_gap_days)
    for column in values.columns:
        distance = (times[column] - ordered["datetime"]).abs()
        values.loc[distance > max_gap, column] = np.nan
    consensus = values.median(axis=1, skipna=True)
    neighbor_count = values.notna().sum(axis=1)
    spread = values.sub(consensus, axis=0).abs().median(axis=1, skipna=True).fillna(np.inf)
    deviation = (ordered["tma_mdpl"] - consensus).abs()
    neighbors_agree = spread <= np.maximum(6.0 * diff_scale, 3.0 * scale)
    isolated = (
        (neighbor_count >= 2) & neighbors_agree
        & (deviation > np.maximum(12.0 * diff_scale, 8.0 * scale))
    )
    nonpositive_reset = (ordered["tma_mdpl"] <= 0.0) & (station_median > 5.0)
    very_extreme = isolated & (deviation > 20.0 * scale)
    ordered["quality_reason"] = np.select(
        [nonpositive_reset, very_extreme, isolated],
        ["nonpositive_reset", "very_extreme_isolated", "isolated_anomaly"],
        default="ok",
    )
    ordered["quality_weight"] = np.select(
        [nonpositive_reset | very_extreme, isolated], [0.05, 0.20], default=1.0
    ).astype(float)
    return ordered[["quality_weight", "quality_reason"]].reindex(frame.index)


def _season_day(datetimes: pd.Series) -> np.ndarray:
    day = datetimes.dt.dayofyear.to_numpy(dtype=float)
    after_february = datetimes.dt.is_leap_year.to_numpy() & (datetimes.dt.month.to_numpy() > 2)
    day[after_february] -= 1.0
    february_29 = (datetimes.dt.month.to_numpy() == 2) & (datetimes.dt.day.to_numpy() == 29)
    day[february_29] = 59.5
    return day


def fit_seasonal_lookup_v2(
    frame: pd.DataFrame, quality_weight: np.ndarray, config: ModelConfig
) -> pd.DataFrame:
    source = frame[["nama_pos", "datetime", "tma_mdpl"]].copy()
    source["hour"] = source["datetime"].dt.hour.astype(int)
    source["season_day_source"] = _season_day(source["datetime"])
    age_days = (source["datetime"].max() - source["datetime"]).dt.total_seconds() / 86400.0
    recency = np.power(0.5, age_days / (365.25 * config.seasonal_recency_half_life_years))
    source["_weight"] = np.asarray(quality_weight, dtype=float) * recency.to_numpy(dtype=float)
    grid = np.arange(1, 366, dtype=float)
    records = []
    for (station, hour), part in source.groupby(["nama_pos", "hour"], observed=True):
        days = part["season_day_source"].to_numpy(dtype=float)
        targets = part["tma_mdpl"].to_numpy(dtype=float)
        base_weight = part["_weight"].to_numpy(dtype=float)
        distance = np.abs(grid[:, None] - days[None, :])
        distance = np.minimum(distance, 365.0 - distance)
        kernel = np.exp(-0.5 * (distance / config.seasonal_bandwidth_days) ** 2)
        weight = kernel * base_weight[None, :]
        denominator = weight.sum(axis=1)
        fallback = float(np.average(targets, weights=np.maximum(base_weight, 1e-12)))
        baseline = np.divide(
            weight @ targets, denominator, out=np.full(365, fallback), where=denominator > 1e-12
        )
        records.append(pd.DataFrame({
            "nama_pos": station, "hour": int(hour),
            "season_day": grid.astype(int), "seasonal_baseline_v2": baseline,
        }))
    return pd.concat(records, ignore_index=True)


def apply_seasonal_lookup_v2(frame: pd.DataFrame, lookup: pd.DataFrame) -> np.ndarray:
    keys = pd.DataFrame({
        "nama_pos": frame["nama_pos"].astype(str).to_numpy(),
        "hour": frame["datetime"].dt.hour.to_numpy(dtype=int),
        "season_day": np.rint(_season_day(frame["datetime"])).astype(int),
    }, index=frame.index)
    indexed = lookup.set_index(["nama_pos", "hour", "season_day"])["seasonal_baseline_v2"]
    query = pd.MultiIndex.from_frame(keys[["nama_pos", "hour", "season_day"]])
    result = indexed.reindex(query).to_numpy(dtype=float)
    if np.isnan(result).any():
        fallback = lookup.groupby("nama_pos", observed=True)["seasonal_baseline_v2"].mean()
        result[np.isnan(result)] = keys.loc[np.isnan(result), "nama_pos"].map(fallback).to_numpy(dtype=float)
    if np.isnan(result).any():
        raise ValueError("Seasonal baseline encountered an unseen station")
    return result


def crossfit_seasonal_baseline_v2(
    frame: pd.DataFrame, diagnostics: pd.DataFrame, config: ModelConfig
) -> np.ndarray:
    date_number = frame["datetime"].dt.floor("D").map(pd.Timestamp.toordinal).to_numpy()
    assignment = date_number % config.seasonal_crossfit_folds
    prediction = np.full(len(frame), np.nan, dtype=float)
    for fold in range(config.seasonal_crossfit_folds):
        fit_mask, hold_mask = assignment != fold, assignment == fold
        lookup = fit_seasonal_lookup_v2(
            frame.loc[fit_mask], diagnostics.loc[fit_mask, "quality_weight"].to_numpy(), config
        )
        prediction[hold_mask] = apply_seasonal_lookup_v2(frame.loc[hold_mask], lookup)
    if np.isnan(prediction).any():
        raise RuntimeError("Cross-fit seasonal baseline contains missing values")
    return prediction


def make_model_v2(
    config: ModelConfig, seed: int, iterations: int, objective: str,
    train_dir: Path | None = None,
):
    try:
        from catboost import CatBoostRegressor
    except ImportError as exc:
        raise RuntimeError("CatBoost is required for training") from exc
    if objective not in {"huber", "rmse"}:
        raise ValueError("objective must be huber or rmse")
    params = dict(
        iterations=iterations, learning_rate=config.learning_rate, depth=config.depth,
        loss_function=(f"Huber:delta={config.huber_delta}" if objective == "huber" else "RMSE"),
        eval_metric=("MAE" if objective == "huber" else "RMSE"),
        l2_leaf_reg=config.l2_leaf_reg, random_seed=seed, random_strength=0.8,
        bootstrap_type="Bayesian", bagging_temperature=0.7, thread_count=-1,
        verbose=config.verbose_every, allow_writing_files=train_dir is not None,
    )
    params.update(catboost_accelerator_params(config))
    if train_dir is not None:
        train_dir.mkdir(parents=True, exist_ok=True)
        params.update(train_dir=str(train_dir), save_snapshot=True,
                      snapshot_file="training.snapshot",
                      snapshot_interval=config.snapshot_interval_seconds)
    return CatBoostRegressor(**params)


def _weighted_rmse(y_true: np.ndarray, y_pred: np.ndarray, weight: np.ndarray) -> float:
    return float(np.sqrt(np.average((np.asarray(y_pred) - np.asarray(y_true)) ** 2, weights=weight)))


def _add_baseline_feature(frame: pd.DataFrame, baseline: np.ndarray) -> pd.DataFrame:
    result = frame.copy()
    result["seasonal_baseline_v2"] = np.asarray(baseline, dtype=float)
    return result


def run_validation_v2(
    train_features: pd.DataFrame, columns: list[str], config: ModelConfig, artifacts: Path,
    checkpoint_handoff: bool = True,
) -> tuple[pd.DataFrame, dict]:
    from catboost import Pool
    artifacts.mkdir(parents=True, exist_ok=True)
    (artifacts / "metrics").mkdir(exist_ok=True)
    (artifacts / "models").mkdir(exist_ok=True)
    (artifacts / "data").mkdir(exist_ok=True)
    records, oof_parts, importance_records = [], [], []
    best_anchor, best_huber, best_rmse = [], [], []
    model_columns = list(columns) + ["seasonal_baseline_v2"]
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in model_columns]
    anchor_cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    for fold_name, train_end, valid_start, valid_end in FOLDS:
        fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
        valid = train_features[train_features["datetime"].between(
            pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both")].copy()
        eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        fit, valid = fit[fit["nama_pos"].isin(eligible)], valid[valid["nama_pos"].isin(eligible)]
        if fit.empty or valid.empty:
            continue
        fit_diag, valid_diag = quality_diagnostics_v2(fit, config), quality_diagnostics_v2(valid, config)
        fit_baseline = crossfit_seasonal_baseline_v2(fit, fit_diag, config)
        lookup = fit_seasonal_lookup_v2(fit, fit_diag["quality_weight"].to_numpy(), config)
        valid_baseline = apply_seasonal_lookup_v2(valid, lookup)
        fit_x, valid_x = _add_baseline_feature(fit, fit_baseline), _add_baseline_feature(valid, valid_baseline)
        stats = fit_target_stats(fit)
        fit_scale = fit["nama_pos"].map(stats["scale"]).to_numpy(dtype=float)
        valid_scale = valid["nama_pos"].map(stats["scale"]).to_numpy(dtype=float)
        y_fit = (fit["tma_mdpl"].to_numpy(dtype=float) - fit_baseline) / fit_scale
        y_valid = (valid["tma_mdpl"].to_numpy(dtype=float) - valid_baseline) / valid_scale
        q_fit = fit_diag["quality_weight"].to_numpy(dtype=float)
        q_valid = valid_diag["quality_weight"].to_numpy(dtype=float)
        rmse_fit_weight = q_fit * fit_scale ** 2
        rmse_fit_weight /= rmse_fit_weight.mean()
        rmse_valid_weight = q_valid * valid_scale ** 2
        rmse_valid_weight /= rmse_valid_weight.mean()
        pools = {
            "huber": (Pool(_xy(fit_x, model_columns), y_fit, cat_features=cat_columns, weight=q_fit),
                      Pool(_xy(valid_x, model_columns), y_valid, cat_features=cat_columns, weight=q_valid)),
            "rmse": (Pool(_xy(fit_x, model_columns), y_fit, cat_features=cat_columns, weight=rmse_fit_weight),
                     Pool(_xy(valid_x, model_columns), y_valid, cat_features=cat_columns, weight=rmse_valid_weight)),
        }
        fold_predictions, fold_iterations = {}, {}
        anchor_train = Pool(_xy(fit, columns), normalize_target(fit, stats),
                            cat_features=anchor_cat_columns, weight=q_fit)
        anchor_valid = Pool(_xy(valid, columns), normalize_target(valid, stats),
                            cat_features=anchor_cat_columns, weight=q_valid)
        anchor_progress = artifacts / "training_progress" / f"v2_{fold_name}_anchor"
        anchor = make_model_v2(config, config.seeds[0], config.default_iterations, "huber", anchor_progress)
        anchor.fit(anchor_train, eval_set=anchor_valid,
                   early_stopping_rounds=config.early_stopping_rounds, use_best_model=True)
        anchor_z = anchor.predict(_xy(valid, columns))
        fold_predictions["anchor"] = denormalize_prediction(anchor_z, valid["nama_pos"], stats)
        fold_iterations["anchor"] = int(anchor.get_best_iteration()) + 1
        anchor.save_model(str(artifacts / "models" / f"v2_validation_{fold_name}_anchor.cbm"))
        for feature, importance in zip(columns, anchor.get_feature_importance()):
            importance_records.append({"fold": fold_name, "component": "anchor",
                                       "feature": feature, "importance": float(importance)})
        for objective in ("huber", "rmse"):
            progress = artifacts / "training_progress" / f"v2_{fold_name}_{objective}"
            model = make_model_v2(config, config.seeds[0], config.default_iterations, objective, progress)
            model.fit(pools[objective][0], eval_set=pools[objective][1],
                      early_stopping_rounds=config.early_stopping_rounds, use_best_model=True)
            residual_z = model.predict(_xy(valid_x, model_columns))
            fold_predictions[objective] = valid_baseline + valid_scale * residual_z
            fold_iterations[objective] = int(model.get_best_iteration()) + 1
            model.save_model(str(artifacts / "models" / f"v2_validation_{fold_name}_{objective}.cbm"))
            for feature, importance in zip(model_columns, model.get_feature_importance()):
                importance_records.append({"fold": fold_name, "component": f"residual_{objective}",
                                           "feature": feature, "importance": float(importance)})
        best_anchor.append(fold_iterations["anchor"])
        best_huber.append(fold_iterations["huber"]); best_rmse.append(fold_iterations["rmse"])
        y_true = valid["tma_mdpl"].to_numpy(dtype=float)
        oof = valid[KEYS + ["tma_mdpl"]].copy()
        oof["fold"] = fold_name; oof["quality_weight"] = q_valid
        oof["quality_reason"] = valid_diag["quality_reason"].to_numpy()
        oof["baseline_prediction"] = valid_baseline
        oof["anchor_prediction"] = fold_predictions["anchor"]
        oof["huber_prediction"] = fold_predictions["huber"]
        oof["rmse_prediction"] = fold_predictions["rmse"]
        oof_parts.append(oof)
        row = {"fold": fold_name, "train_rows": len(fit), "valid_rows": len(valid),
               "quality_downweighted_train": int((q_fit < 1).sum()),
               "quality_downweighted_valid": int((q_valid < 1).sum()),
               "anchor_best_iteration": fold_iterations["anchor"],
               "huber_best_iteration": fold_iterations["huber"],
               "rmse_best_iteration": fold_iterations["rmse"]}
        for name, pred in [("baseline", valid_baseline), ("anchor", fold_predictions["anchor"]),
                           ("huber", fold_predictions["huber"]),
                           ("rmse", fold_predictions["rmse"])]:
            metric = metrics_frame(y_true, pred, valid["nama_pos"])
            row.update({f"{name}_{key}": value for key, value in metric.items()})
            row[f"{name}_quality_rmse"] = _weighted_rmse(y_true, pred, q_valid)
        records.append(row)
        LOGGER.info("V2 fold %s complete: Huber %.4f, RMSE %.4f", fold_name, row["huber_rmse"], row["rmse_rmse"])
        pd.concat(oof_parts, ignore_index=True).to_csv(artifacts / "metrics" / "v2_oof_predictions.csv", index=False)
        pd.DataFrame(records).to_csv(artifacts / "metrics" / "v2_validation_metrics.csv", index=False)
        pd.DataFrame(importance_records).to_csv(artifacts / "metrics" / "v2_feature_importance.csv", index=False)
        if checkpoint_handoff and (artifacts / "manifest.json").exists():
            update_manifest(artifacts, stage=f"v2_validation_{fold_name}_complete")
            create_handoff_zip(artifacts, "in_progress")
    if not records:
        raise RuntimeError("No V2 validation fold was runnable")
    oof_all = pd.concat(oof_parts, ignore_index=True)
    blend_records = []
    for alpha in np.linspace(0.0, 1.0, config.blend_grid_size):
        residual_pred = (1.0 - alpha) * oof_all["huber_prediction"] + alpha * oof_all["rmse_prediction"]
        for beta in np.linspace(0.0, 1.0, config.blend_grid_size):
            pred = (1.0 - beta) * oof_all["anchor_prediction"] + beta * residual_pred
            fold_scores = []
            for _, part in oof_all.assign(_prediction=pred).groupby("fold", observed=True):
                fold_scores.append(_weighted_rmse(part["tma_mdpl"], part["_prediction"], part["quality_weight"]))
            blend_records.append({"rmse_fraction": float(alpha), "residual_fraction": float(beta),
                                  "mean_fold_quality_rmse": float(np.mean(fold_scores)),
                                  "std_fold_quality_rmse": float(np.std(fold_scores)),
                                  "selection_score": float(np.mean(fold_scores) + 0.1 * np.std(fold_scores)),
                                  "pooled_raw_rmse": float(mean_squared_error(oof_all["tma_mdpl"], pred) ** 0.5)})
    blend_search = pd.DataFrame(blend_records).sort_values(
        ["selection_score", "residual_fraction", "rmse_fraction"]
    )
    alpha = float(blend_search.iloc[0]["rmse_fraction"])
    beta = float(blend_search.iloc[0]["residual_fraction"])
    residual_pred = (1.0 - alpha) * oof_all["huber_prediction"] + alpha * oof_all["rmse_prediction"]
    oof_all["prediction"] = (1.0 - beta) * oof_all["anchor_prediction"] + beta * residual_pred
    oof_all["error"] = oof_all["prediction"] - oof_all["tma_mdpl"]
    oof_all["absolute_error"] = oof_all["error"].abs()
    report = pd.DataFrame(records)
    for index, row in report.iterrows():
        part = oof_all[oof_all["fold"] == row["fold"]]
        metric = metrics_frame(part["tma_mdpl"], part["prediction"], part["nama_pos"])
        for key, value in metric.items(): report.loc[index, f"blend_{key}"] = value
        report.loc[index, "blend_quality_rmse"] = _weighted_rmse(part["tma_mdpl"], part["prediction"], part["quality_weight"])
    station_metrics = oof_all.groupby(["fold", "nama_pos"], observed=True).agg(
        rows=("error", "size"), mae=("absolute_error", "mean"),
        rmse=("error", lambda s: float(np.sqrt(np.mean(s ** 2)))), bias=("error", "mean")
    ).reset_index()
    tuning = {"version": "v2", "blend_rmse_fraction": alpha,
              "blend_residual_fraction": beta,
              "anchor_iterations": int(np.median(best_anchor)),
              "huber_iterations": int(np.median(best_huber)),
              "rmse_iterations": int(np.median(best_rmse)),
              "selection_rule": "joint anchor/residual/objective grid; mean fold quality RMSE + 0.1 * fold std; OOF only"}
    report.to_csv(artifacts / "metrics" / "v2_validation_metrics.csv", index=False)
    oof_all.to_csv(artifacts / "metrics" / "v2_oof_predictions.csv", index=False)
    station_metrics.to_csv(artifacts / "metrics" / "v2_fold_station_metrics.csv", index=False)
    blend_search.to_csv(artifacts / "metrics" / "v2_blend_search.csv", index=False)
    pd.DataFrame(importance_records).to_csv(artifacts / "metrics" / "v2_feature_importance.csv", index=False)
    write_json(artifacts / "v2_tuning.json", tuning)
    full_quality = quality_diagnostics_v2(train_features, config)
    train_features[KEYS + ["tma_mdpl"]].join(full_quality).to_csv(
        artifacts / "data" / "v2_quality_diagnostics.csv", index=False)
    if checkpoint_handoff and (artifacts / "manifest.json").exists():
        update_manifest(artifacts, stage="v2_validation_complete", v2_tuning=tuning)
        create_handoff_zip(artifacts, "in_progress")
    return report, tuning


def find_latest_v2_tuning(output_dir: Path) -> dict | None:
    candidates = sorted(output_dir.glob("handoff_*/v2_tuning.json"),
                        key=lambda path: path.stat().st_mtime, reverse=True)
    return json.loads(candidates[0].read_text(encoding="utf-8")) if candidates else None


def fit_final_and_submit_v2(
    train_features: pd.DataFrame, test_features: pd.DataFrame, sample: pd.DataFrame,
    columns: list[str], config: ModelConfig, artifacts: Path, tuning: dict,
) -> Path:
    from catboost import Pool
    diagnostics = quality_diagnostics_v2(train_features, config)
    quality = diagnostics["quality_weight"].to_numpy(dtype=float)
    train_baseline = crossfit_seasonal_baseline_v2(train_features, diagnostics, config)
    lookup = fit_seasonal_lookup_v2(train_features, quality, config)
    test_baseline = apply_seasonal_lookup_v2(test_features, lookup)
    train_x = _add_baseline_feature(train_features, train_baseline)
    test_x = _add_baseline_feature(test_features, test_baseline)
    model_columns = list(columns) + ["seasonal_baseline_v2"]
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in model_columns]
    stats = fit_target_stats(train_features)
    scale = train_features["nama_pos"].map(stats["scale"]).to_numpy(dtype=float)
    test_scale = test_features["nama_pos"].map(stats["scale"]).to_numpy(dtype=float)
    y = (train_features["tma_mdpl"].to_numpy(dtype=float) - train_baseline) / scale
    final_importance_records = []
    anchor_cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    anchor_pool = Pool(_xy(train_features, columns), normalize_target(train_features, stats),
                       cat_features=anchor_cat_columns, weight=quality)
    anchor_predictions = []
    for seed in config.seeds:
        progress = artifacts / "training_progress" / f"v2_final_anchor_seed_{seed}"
        anchor = make_model_v2(config, seed, int(tuning["anchor_iterations"]), "huber", progress)
        anchor.fit(anchor_pool)
        anchor.save_model(str(artifacts / "models" / f"v2_anchor_seed_{seed}.cbm"))
        for feature, importance in zip(columns, anchor.get_feature_importance()):
            final_importance_records.append({"seed": seed, "component": "anchor",
                                             "feature": feature, "importance": float(importance)})
        anchor_z = anchor.predict(_xy(test_features, columns))
        anchor_predictions.append(denormalize_prediction(anchor_z, test_features["nama_pos"], stats))
        if (artifacts / "manifest.json").exists():
            update_manifest(artifacts, stage=f"v2_final_anchor_seed_{seed}_complete")
            create_handoff_zip(artifacts, "in_progress")
    anchor_prediction = np.mean(anchor_predictions, axis=0)
    objective_predictions = {}
    for objective in ("huber", "rmse"):
        weight = quality if objective == "huber" else quality * scale ** 2
        weight = weight / weight.mean()
        train_pool = Pool(_xy(train_x, model_columns), y, cat_features=cat_columns, weight=weight)
        seed_predictions = []
        for seed in config.seeds:
            iterations = int(tuning[f"{objective}_iterations"])
            progress = artifacts / "training_progress" / f"v2_final_{objective}_seed_{seed}"
            model = make_model_v2(config, seed, iterations, objective, progress)
            model.fit(train_pool)
            model.save_model(str(artifacts / "models" / f"v2_{objective}_seed_{seed}.cbm"))
            for feature, importance in zip(model_columns, model.get_feature_importance()):
                final_importance_records.append({"seed": seed, "component": f"residual_{objective}",
                                                 "feature": feature, "importance": float(importance)})
            seed_predictions.append(test_baseline + test_scale * model.predict(_xy(test_x, model_columns)))
            if (artifacts / "manifest.json").exists():
                update_manifest(artifacts, stage=f"v2_final_{objective}_seed_{seed}_complete")
                create_handoff_zip(artifacts, "in_progress")
        objective_predictions[objective] = np.mean(seed_predictions, axis=0)
    alpha = float(tuning["blend_rmse_fraction"])
    beta = float(tuning["blend_residual_fraction"])
    residual_prediction = (1.0 - alpha) * objective_predictions["huber"] + alpha * objective_predictions["rmse"]
    prediction = (1.0 - beta) * anchor_prediction + beta * residual_prediction
    ordered = pd.DataFrame({"_row_order": test_features["_row_order"],
                            "id": test_features["id"], "tma_mdpl": prediction}).sort_values("_row_order")
    submission = sample[["id"]].merge(ordered[["id", "tma_mdpl"]], on="id", how="left", validate="one_to_one")
    if submission["tma_mdpl"].isna().any() or not np.isfinite(submission["tma_mdpl"]).all():
        raise ValueError("V2 submission contains missing/non-finite predictions")
    output = artifacts / "submission.csv"
    submission.to_csv(output, index=False)
    lookup.to_csv(artifacts / "data" / "v2_seasonal_lookup.csv", index=False)
    train_features[KEYS + ["tma_mdpl"]].join(diagnostics).to_csv(
        artifacts / "data" / "v2_quality_diagnostics.csv", index=False)
    pd.DataFrame(final_importance_records).to_csv(
        artifacts / "metrics" / "v2_final_feature_importance.csv", index=False)
    write_json(artifacts / "v2_tuning.json", tuning)
    write_json(artifacts / "v2_model_metadata.json", {
        "architecture": "direct Huber anchor + cross-fit seasonal residual Huber/RMSE blend",
        "model_columns": model_columns, "seeds": list(config.seeds),
        "quality_reason_counts": diagnostics["quality_reason"].value_counts().to_dict(),
        "tuning": tuning,
    })
    return output


## 11. Orkestrator notebook dan finalisasi handoff


In [ ]:
def find_latest_recommended_iterations(output_dir: Path) -> int | None:
    candidates = sorted(
        output_dir.glob("handoff_*/recommended_iterations.txt"),
        key=lambda path: path.stat().st_mtime, reverse=True,
    )
    return int(candidates[0].read_text(encoding="utf-8").strip()) if candidates else None


def run_pipeline(
    mode: str = "audit",
    data_dir: Path = Path("datasets"),
    output_dir: Path = Path("artifacts"),
    iterations: int | None = None,
    save_feature_matrices: bool = True,
    task_type: str = "CPU",
    devices: str | None = None,
    notebook_path: Path | None = None,
) -> dict:
    """Run a Kaggle/local mode and create a timestamped diagnostic handoff."""
    if mode not in {"audit", "validate", "fit", "all"}:
        raise ValueError("mode must be one of: audit, validate, fit, all")
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        force=True,
    )
    paths = Paths(root=Path(data_dir), artifacts=Path(output_dir))
    config = ModelConfig(task_type=task_type, devices=devices)
    train, test, sample, coordinates, environment = load_inputs(paths)
    audit = audit_inputs(train, test, sample, coordinates, environment)
    LOGGER.info("Input audit passed:\n%s", json.dumps(audit, indent=2))
    result = {"mode": mode, "audit": audit}
    if mode == "audit":
        return result

    accelerator = validate_accelerator(config)
    result["accelerator"] = accelerator
    LOGGER.info("Compute backend: %s", json.dumps(accelerator))

    run_dir = create_run_directory(paths.artifacts, mode)
    attach_run_logger(run_dir)
    result["run_id"] = run_dir.name
    result["run_dir"] = str(run_dir)
    source_provenance = notebook_provenance(notebook_path)
    if source_provenance["status"] != "hashed":
        LOGGER.warning("Notebook file unavailable; using declared canonical source hash")
    effective_config = {
        "model": asdict(config), "folds": FOLDS, "mode": mode,
        "requested_iterations": iterations,
        "save_feature_matrices": save_feature_matrices,
        "accelerator": accelerator,
    }
    manifest = {
        "schema_version": HANDOFF_SCHEMA_VERSION,
        "run_id": run_dir.name,
        "mode": mode,
        "status": "in_progress",
        "stage": "initialized",
        "started_at_utc": utc_now_iso(),
        "input_dir": str(paths.root),
        "output_root": str(paths.artifacts),
        "notebook_sha256": (source_provenance["sha256"]
                              or source_provenance["source_sha256"]),
        "notebook_hash_kind": ("raw_file_sha256" if source_provenance["sha256"]
                               else "canonical_source_sha256"),
        "code_provenance": source_provenance,
        "effective_config": effective_config,
        "effective_model": None,
        "core_handoff_files": [
            "manifest.json", "config.json", "runtime_versions.json",
            "input_inventory.json", "data/train_features.*",
            "data/test_features.*", "data/train_sample_diagnostics.*",
            "metrics/environment_drift.csv", "run.log",
        ],
        "mode_dependent_files": [
            "v2_tuning.json", "metrics/v2_oof_predictions.csv",
            "metrics/v2_validation_metrics.csv", "metrics/v2_blend_search.csv",
            "metrics/v2_fold_station_metrics.csv", "data/v2_quality_diagnostics.csv",
            "metrics/v2_feature_importance.csv", "metrics/v2_final_feature_importance.csv",
            "data/v2_seasonal_lookup.csv", "v2_model_metadata.json",
            "training_progress/", "models/", "submission.csv",
        ],
    }
    write_json(run_dir / "manifest.json", manifest)
    write_json(run_dir / "data_audit.json", audit)
    write_json(run_dir / "runtime_versions.json", runtime_versions())
    write_json(run_dir / "input_inventory.json", {"files": input_inventory(paths)})
    write_json(run_dir / "config.json", effective_config)
    write_handoff_readme(run_dir)

    try:
        train_features, test_features, upstream_weights = build_feature_tables(
            train, test, coordinates, environment, paths, config
        )
        columns = feature_columns(train_features)
        save_pretraining_handoff(
            run_dir, train_features, test_features, columns, upstream_weights,
            save_feature_matrices=save_feature_matrices,
        )
        LOGGER.info("Built %d leakage-safe features", len(columns))
        result["feature_count"] = len(columns)
        tuning = None
        if mode in {"validate", "all"}:
            update_manifest(
                run_dir, stage="validation_training_started",
                training_started_at_utc=utc_now_iso(),
            )
            report, tuning = run_validation_v2(
                train_features, columns, config, run_dir
            )
            result["validation"] = report
            result["v2_tuning"] = tuning
            update_manifest(
                run_dir, effective_model=effective_v2_model_manifest(
                    config, tuning, "current_validation"
                ),
            )
            LOGGER.info("Validation report:\n%s", report.to_string(index=False))
        if mode in {"fit", "all"}:
            prior_tuning = find_latest_v2_tuning(paths.artifacts)
            if tuning is not None:
                final_tuning, tuning_source = dict(tuning), "current_validation"
            elif prior_tuning is not None:
                final_tuning, tuning_source = dict(prior_tuning), "latest_prior_validation"
            else:
                final_tuning = {"version": "v2", "blend_rmse_fraction": 0.5,
                                "blend_residual_fraction": 0.5,
                                "anchor_iterations": config.default_iterations,
                                "huber_iterations": config.default_iterations,
                                "rmse_iterations": config.default_iterations,
                                "selection_rule": "fallback; run validate before fit"}
                tuning_source = "config_fallback"
            if iterations is not None:
                final_tuning["anchor_iterations"] = int(iterations)
                final_tuning["huber_iterations"] = int(iterations)
                final_tuning["rmse_iterations"] = int(iterations)
                tuning_source = f"{tuning_source}+explicit_iterations"
            update_manifest(
                run_dir, stage="final_training_started",
                final_training_started_at_utc=utc_now_iso(),
                tuning_source=tuning_source, v2_tuning=final_tuning,
                effective_model=effective_v2_model_manifest(
                    config, final_tuning, tuning_source
                ),
            )
            output = fit_final_and_submit_v2(
                train_features, test_features, sample, columns, config,
                run_dir, final_tuning,
            )
            result["submission"] = str(output)
            result["v2_tuning"] = final_tuning
            result["tuning_source"] = tuning_source
            LOGGER.info("Submission written to %s", output)
        result["handoff_zip"] = str(paths.artifacts / f"{run_dir.name}_complete.zip")
        result_summary = {
            key: (value.to_dict(orient="records") if isinstance(value, pd.DataFrame) else value)
            for key, value in result.items()
        }
        write_json(run_dir / "result_summary.json", result_summary)
        update_manifest(
            run_dir, status="complete", stage="complete",
            completed_at_utc=utc_now_iso(), handoff_zip=result["handoff_zip"],
            artifact_sha256=core_artifact_hashes(run_dir),
        )
        handoff_zip = create_handoff_zip(run_dir, "complete")
        result["handoff_zip"] = str(handoff_zip)
        return result
    except Exception:
        error_text = traceback.format_exc()
        (run_dir / "error_traceback.txt").write_text(error_text, encoding="utf-8")
        LOGGER.exception("Pipeline failed; preserving partial handoff")
        update_manifest(
            run_dir, status="failed", stage="failed",
            failed_at_utc=utc_now_iso(),
        )
        create_handoff_zip(run_dir, "failed")
        raise


## 12. Pilih mode eksekusi

- `audit`: validasi input saja, tanpa training.
- `validate`: preprocessing dan cross-validation, tanpa final fit.
- `fit`: final fit dan submission.
- `all`: validasi, final fit, submission, dan ZIP handoff lengkap.

Pada Kaggle, input otomatis memakai `/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026` dan seluruh output memakai `/kaggle/working`. Aktifkan **Accelerator: GPU T4 x2** pada Settings; seluruh model CatBoost memakai perangkat `0:1`, `boosting_type=Plain`, `border_count=254`, `metric_period=5`, dan partisi data multi-GPU. Parameter tersebut dipin agar seluruh fold memakai algoritme yang sama dengan final fit.


In [ ]:
# Guard: Run All hanya melakukan audit dan TIDAK melatih model.
KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026")
ON_KAGGLE = KAGGLE_DATA_DIR.exists()

MODE = "audit"  # pilihan: audit, validate, fit, all
DATA_DIR = KAGGLE_DATA_DIR if ON_KAGGLE else Path("datasets")
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts")
ITERATIONS = None  # None = rekomendasi validasi atau default konfigurasi
SAVE_FEATURE_MATRICES = True  # disarankan untuk handoff penyempurnaan
TASK_TYPE = "GPU" if ON_KAGGLE else "CPU"
GPU_DEVICES = "0:1" if ON_KAGGLE else None  # kedua unit T4 Kaggle
NOTEBOOK_PATH = Path("notebook-150726.ipynb")
if not NOTEBOOK_PATH.is_file():
    NOTEBOOK_PATH = None  # manifest menandai hash unavailable secara eksplisit

print(f"Runtime: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Input : {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Compute: {TASK_TYPE} ({GPU_DEVICES or 'default device'})")
print("CatBoost: boosting_type=Plain, border_count=254, metric_period=5")


## 13. Jalankan mode yang dipilih


In [ ]:
result = run_pipeline(
    mode=MODE,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    iterations=ITERATIONS,
    save_feature_matrices=SAVE_FEATURE_MATRICES,
    task_type=TASK_TYPE,
    devices=GPU_DEVICES,
    notebook_path=NOTEBOOK_PATH,
)
result
